In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import math


Carga del dataset

In [ ]:
datos, metadatos = tfds.load('cifar10', as_supervised=True, with_info=True)

CLASE_LIVIANO = 1  # automobile
CLASE_PESADO  = 9  # truck

nombres_clases = ['Liviano', 'Pesado']

print("Clases originales de CIFAR-10:")
print(metadatos.features['label'].names)

Filtrado y reasignación de etiquetas

In [ ]:
def filtrar_vehiculos(imagen, etiqueta):
    return tf.logical_or(
        tf.equal(etiqueta, CLASE_LIVIANO),
        tf.equal(etiqueta, CLASE_PESADO)
    )

def remap_etiquetas(imagen, etiqueta):
    nueva_etiqueta = tf.cast(tf.equal(etiqueta, CLASE_PESADO), tf.int64)
    return imagen, nueva_etiqueta

datos_entrenamiento_raw = datos['train'].filter(filtrar_vehiculos).map(remap_etiquetas)
datos_pruebas_raw       = datos['test'].filter(filtrar_vehiculos).map(remap_etiquetas)

print("Dataset filtrado: automobile vs truck")

Normalización

In [ ]:
def normalizar(imagenes, etiquetas):
    imagenes = tf.cast(imagenes, tf.float32)
    imagenes /= 255.0
    return imagenes, etiquetas

Visualización: imagen suelta

In [ ]:
for imagen, etiqueta in datos_entrenamiento_raw.take(1):
    break

plt.figure()
plt.imshow(imagen.numpy())
plt.colorbar()
plt.grid(False)
plt.title(nombres_clases[etiqueta.numpy()])
plt.show()

Visualización: grilla de 5x5

In [ ]:
plt.figure(figsize=(10, 10))
for i, (imagen, etiqueta) in enumerate(datos_entrenamiento_raw.take(25)):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(imagen.numpy())
    plt.xlabel(nombres_clases[etiqueta.numpy()])
plt.suptitle("Dataset filtrado", fontsize=14)
plt.tight_layout()
plt.show()

Modelo

In [ ]:
modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3), dtype=tf.float32),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation=tf.nn.relu),
    tf.keras.layers.Dense(128, activation=tf.nn.relu),
    tf.keras.layers.Dense(2, activation=tf.nn.softmax)
])

modelo.summary()

Compilación

In [ ]:
modelo.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

Batchs de entrenamiento

In [ ]:
ej_entrenamiento = 10000
ej_pruebas       = 2000
LOTE = 32

datos_entrenamiento = (
    datos_entrenamiento_raw
    .map(normalizar)
    .cache()
    .shuffle(ej_entrenamiento)
    .batch(LOTE)
    .repeat()
)

datos_pruebas = (
    datos_pruebas_raw
    .map(normalizar)
    .cache()
    .batch(LOTE)
)

Entrenamiento

In [ ]:
historial = modelo.fit(
    datos_entrenamiento,
    epochs=10,
    steps_per_epoch=math.ceil(ej_entrenamiento / LOTE)
)

Curvas de entrenamiento

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.xlabel("# Época")
plt.ylabel("Magnitud de pérdida")
plt.plot(historial.history["loss"])
plt.title("Pérdida durante el entrenamiento")

plt.subplot(1, 2, 2)
plt.xlabel("# Época")
plt.ylabel("Precisión")
plt.plot(historial.history["accuracy"])
plt.title("Precisión durante el entrenamiento")

plt.tight_layout()
plt.show()

Evaluación

In [ ]:
perdida, precision = modelo.evaluate(datos_pruebas)
print(f"\nPérdida en pruebas:   {perdida:.4f}")
print(f"Precisión en pruebas: {precision*100:.2f}%")

Lote para predicciones

In [ ]:
for imagenes_prueba, etiquetas_prueba in datos_pruebas.take(1):
    imagenes_prueba  = imagenes_prueba.numpy()
    etiquetas_prueba = etiquetas_prueba.numpy()
    predicciones     = modelo.predict(imagenes_prueba)

Predicción imagen suelta

In [ ]:
imagen_idx    = 4
imagen_suelta = imagenes_prueba[imagen_idx]
imagen_suelta = np.array([imagen_suelta])

prediccion = modelo.predict(imagen_suelta)

print("Imagen real:    ", nombres_clases[etiquetas_prueba[imagen_idx]])
print("Predicción:     ", nombres_clases[np.argmax(prediccion[0])])
print(f"  Liviano: {prediccion[0][0]*100:.1f}%")
print(f"  Pesado:  {prediccion[0][1]*100:.1f}%")

plt.figure(figsize=(2, 2))
plt.imshow(imagenes_prueba[imagen_idx])
plt.title(f"Predicción: {nombres_clases[np.argmax(prediccion[0])]}")
plt.axis('off')
plt.show()